In [1]:
import pandas as pd

# 1) read excel file
df = pd.read_excel("01M851A81RPR7NYATMTEYT6K5H.xlsx")

In [2]:
# 2) Remove extra spaces from all text columns
for col in ["Customer", "Product", "City"]:
    df[col] = df[col].str.strip()

In [3]:
# 3) Standardize cities (Arabic + English -> English)
cities = {
    "cairo": "Cairo", "القاهرة": "Cairo",
    "giza": "Giza", "الجيزة": "Giza",
    "alexandria": "Alexandria", "الإسكندرية": "Alexandria",
    "sharqia": "Sharqia", "الشرقية": "Sharqia",
}
df["City"] = df["City"].str.lower().map(cities).fillna("Unknown")

In [4]:
# 4) Standardize dates to YYYY-MM-DD
def fix_date(d):
    d = d.strip().replace("/", "-").replace(".", "-")
    if d[:4].isdigit() and len(d.split("-")[0]) == 4:   # 2026-11-21
        return pd.to_datetime(d, format="%Y-%m-%d")
    return pd.to_datetime(d, format="%d-%m-%Y")          # 25/10/2026

df["Order Date"] = df["Order Date"].apply(fix_date)

In [5]:
# 5) Missing price: fill it from the same product (each product has a fixed price)
df["Unit Price (EGP)"] = df["Unit Price (EGP)"].fillna(
    df.groupby("Product")["Unit Price (EGP)"].transform("max"))

In [6]:
# 6) delete duplicated rows
before = len(df)
df = df.drop_duplicates()
removed = before - len(df)

In [7]:
# 7) Missing quantity: leave it empty (we can't guess it) and flag it
df["Missing Quantity"] = df["Quantity"].isna().map({True: "Yes", False: ""})

In [8]:
# 8) Order total + sorting
df["Total (EGP)"] = df["Quantity"] * df["Unit Price (EGP)"]
df = df.sort_values("Order ID").reset_index(drop=True)

In [9]:
# 9) summary
summary = pd.DataFrame({
    "Item": ["Rows before", "Duplicates removed", "Rows after",
             "Missing quantity (left blank)", "Missing city (set Unknown)",
             "Total sales (EGP)"],
    "Value": [before, removed, len(df),
              int(df["Quantity"].isna().sum()), int((df["City"] == "Unknown").sum()),
              df["Total (EGP)"].sum()],
})

In [10]:
# 10) save cleaned file
with pd.ExcelWriter("cleaned_sales.xlsx", date_format="YYYY-MM-DD") as w:
    df.to_excel(w, sheet_name="Cleaned", index=False)
    summary.to_excel(w, sheet_name="Summary", index=False)

print(summary)

                            Item     Value
0                    Rows before      41.0
1             Duplicates removed       3.0
2                     Rows after      38.0
3  Missing quantity (left blank)       2.0
4     Missing city (set Unknown)       1.0
5              Total sales (EGP)  689550.0
